# Lab 11: Automated Evaluation with Cloud Evaluators

> **Source:** Microsoft Learning -- [mslearn-genaiops](https://github.com/MicrosoftLearning/mslearn-genaiops), `docs/04-automated-evaluation.md`, `src/evaluators/evaluate_agent.py`
> **License:** MIT

## Objectives

By the end of this lab you will:

1. Understand the **evaluation dataset** structure (89-item JSONL)
2. Configure **cloud evaluators**: IntentResolution, Relevance, Groundedness
3. Run an **automated evaluation** pipeline
4. Set up **GitHub Actions with OIDC** federated credentials for CI/CD evaluation

| Estimated Time | Estimated Cost |
|---|---|
| ~40 minutes | ~$5--10 (cloud evaluators use LLM-as-judge) |

## Architecture

```mermaid
graph LR
    DATASET[89-item JSONL] --> EVAL[Cloud Evaluators]
    EVAL --> IR[IntentResolution<br/>Evaluator]
    EVAL --> REL[Relevance<br/>Evaluator]
    EVAL --> GND[Groundedness<br/>Evaluator]
    IR --> SCORES[Aggregate Scores]
    REL --> SCORES
    GND --> SCORES
    GH[GitHub Actions] -->|OIDC auth| EVAL
```

Cloud evaluators are **LLM-as-judge** services hosted in Azure AI Foundry. They use a separate LLM to score your agent's responses on standardized rubrics -- the same dimensions we scored manually in Lab 10, now automated at scale.

## The Evaluation Dataset

The evaluation dataset is an **89-item JSONL** file where each line is a JSON object with query, expected context, and (optionally) a reference response.

The 89 items cover five categories:

| Category | Count | Purpose |
|---|---|---|
| Simple factual | ~20 | "What gear for a day hike?" -- tests basic knowledge retrieval |
| Complex planning | ~25 | "Plan a 3-day route in Yosemite" -- tests multi-step reasoning |
| Safety-critical | ~15 | "Bear encounter protocol" -- tests safety guardrails |
| Ambiguous | ~15 | "What's the best trail?" (no context) -- tests clarification behavior |
| Out-of-scope | ~14 | "What's the weather in Paris?" -- tests boundary enforcement |

> **Exam Tip:** 89 items provides **statistical significance** for aggregate scores. Fewer than ~30 items makes averages unreliable; more than ~100 increases cost without proportional benefit for development-stage evaluation.

## The 5-Step Evaluation Pipeline

The full evaluation script (`scripts/evaluate_agent.py`) follows this pipeline:

### Step 1: Upload Dataset
Upload the JSONL file to Azure AI Foundry as a registered dataset.

### Step 2: Create Evaluation Definition
Define which evaluators to use and how dataset columns map to evaluator inputs:

```python
evaluators = {
    "intent_resolution": {"id": "azureai://intent-resolution-evaluator"},
    "relevance":         {"id": "azureai://relevance-evaluator"},
    "groundedness":      {"id": "azureai://groundedness-evaluator"},
}

data_mapping = {
    "intent_resolution": {"query": "${data.query}", "response": "${data.response}"},
    "relevance":         {"query": "${data.query}", "response": "${data.response}"},
    "groundedness":      {"query": "${data.query}", "response": "${data.response}", "context": "${data.context}"},
}
```

The `${data.field_name}` syntax binds **dataset columns** to **evaluator inputs**. This is how the platform knows which column is the query, which is the response, etc.

### Step 3: Run Evaluation
Submit the evaluation job to Azure AI Foundry.

### Step 4: Poll for Results
The evaluation runs asynchronously. Poll every 30 seconds until status is `Completed`, `Failed`, or `Canceled`.

### Step 5: Retrieve Scores
Fetch per-item scores and aggregate metrics.

> **Exam Tip:** `data_mapping` with `${data.field_name}` syntax is a key concept. It decouples your dataset schema from evaluator expectations -- you can use any column names in your JSONL as long as the mapping is correct.

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

load_dotenv()
credential = DefaultAzureCredential()
project_client = AIProjectClient(
    credential=credential,
    endpoint=os.getenv("AZURE_AI_PROJECT_ENDPOINT"),
)

# This is a simplified version -- the full script is in scripts/evaluate_agent.py
print("Run the full evaluation with: python scripts/evaluate_agent.py")
print("Expected V4 scores: Intent 4.3/5, Relevance 4.5/5, Groundedness 4.1/5")

## GitHub Actions with OIDC Federated Credentials

To run evaluations automatically on every push to `main`, use **GitHub Actions** with **OIDC (OpenID Connect) federated credentials**.

### Why OIDC?

| Approach | Drawback |
|---|---|
| Long-lived secrets (`AZURE_CLIENT_SECRET`) | Must be rotated manually; risk of leakage |
| **OIDC federated credentials** | **No secrets to rotate**; token issued per-run; scoped to branch |

### Setup Steps

1. **Azure AD App Registration** -- create an app registration in Entra ID
2. **Federated Credential** -- add a federated credential scoped to `repo:<org>/<repo>:ref:refs/heads/main`
3. **GitHub Secrets** -- store `AZURE_CLIENT_ID`, `AZURE_TENANT_ID`, `AZURE_SUBSCRIPTION_ID` (these are IDs, not secrets)
4. **Workflow** -- use `azure/login@v2` with OIDC:

```yaml
- name: Azure Login (OIDC)
  uses: azure/login@v2
  with:
    client-id: ${{ env.AZURE_CLIENT_ID }}
    tenant-id: ${{ env.AZURE_TENANT_ID }}
    subscription-id: ${{ env.AZURE_SUBSCRIPTION_ID }}
```

See the full workflow in `workflows/evaluate-agent.yml`.

> **Exam Tip:** OIDC federated credentials are the **recommended** approach for CI/CD authentication to Azure. No secret rotation, scoped to specific branches, and tokens are short-lived. The exam may contrast this with long-lived service principal secrets.

## Review Results in Azure Portal

After the evaluation completes, view results in the **Azure AI Foundry portal**:

1. Navigate to [ai.azure.com](https://ai.azure.com) and open your project
2. Go to **Evaluation** in the left navigation
3. Click on the evaluation run to see:
   - **Aggregate scores** -- average per evaluator across all 89 items
   - **Per-item scores** -- drill into individual responses to understand failures
   - **Score distributions** -- histograms showing the spread of scores
   - **Trends** -- compare across multiple evaluation runs over time

## Key Takeaways

| Concept | Detail |
|---|---|
| **Cloud Evaluators** | IntentResolution, Relevance, Groundedness -- LLM-as-judge services that automate the manual scoring from Lab 10. |
| **`data_mapping` Syntax** | `${data.field_name}` binds dataset columns to evaluator inputs. Decouples schema from evaluator expectations. |
| **OIDC Federated Credentials** | No long-lived secrets. GitHub Actions gets a short-lived token per run, scoped to a specific branch. |
| **LLM-as-Judge Cost** | Cloud evaluators use an LLM to score each item -- this has its own cost (~$5--10 for 89 items x 3 evaluators). Budget for this. |
| **Async Pipeline** | Evaluations run asynchronously. Poll for completion or use webhooks in production. |

---

**Next:** [Lab 12 -- Monitoring and Tracing](../lab12-monitoring-tracing/lab12-monitoring-tracing.ipynb) -- instrument with OpenTelemetry and analyze in Azure Monitor.